# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Selected lane: Refresh / Content Opportunity Scoring (core lane).** This is the most practical Week 1 starting point because the guide explicitly pairs it with the shipped starter dataset, and the CSV has one row per content item plus 90-day search, engagement, freshness, and content-size signals. The repository already includes a runnable refresh baseline and ranked review queue, so I can begin with an explainable score before deciding whether a more complex model earns its place. I can work on the starter slice now and use the warehouse later if a future-window outcome is needed. The goal is to prioritize pages for an editor to review, not to automatically prescribe or publish changes.

In [6]:
from pathlib import Path
import pandas as pd

DATA_FILE = "data/raw/content_refresh_anonymized.csv"
repository_roots = [Path.cwd(), *Path.cwd().parents]
data_path = next(
    (root / DATA_FILE for root in repository_roots if (root / DATA_FILE).is_file()),
    None,
)
if data_path is None:
    raise FileNotFoundError(f"Could not find {DATA_FILE} from {Path.cwd()}")

df = pd.read_csv(data_path)
required_columns = {
    "content_id",
    "client_id",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "sessions_90d",
    "engagement_rate",
    "word_count",
    "trend_direction",
    "trend_pct",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Starter CSV is missing required columns: {sorted(missing_columns)}")

## 2. The question: decision, action, cost of a wrong call

- **Decision:** Given limited editorial capacity, which eligible content item should be reviewed first for a possible refresh?
- **Unit of analysis:** One pseudonymized content item/page at the point the review queue is made.
- **Action:** An editor reviews a high-ranked page, checks its reason codes and source metrics, then decides whether to refresh it, investigate it further, or leave it unchanged.
- **Cost of a wrong call:** A false positive spends scarce editor time on a page that does not need work; a false negative leaves a useful improvement opportunity unreviewed. The score should order review, not trigger an edit.

This is a **ranking / scoring** task: the first useful output is a priority order with understandable reasons, evaluated at a realistic review capacity (for example, precision among the top K reviewed pages). The starter snapshot supports an initial rule-based queue, but it does not establish a future outcome by itself.

In [7]:
decision_frame = {
    "decision": "Which content item should an editor review first?",
    "unit": "One pseudonymized content item/page at queue time.",
    "action": "Review the page and its reason codes; choose refresh, investigate, or no change.",
    "wrong_call_cost": "Wasted editor time for a false positive; a missed improvement opportunity for a false negative.",
}
assert all(value.strip() for value in decision_frame.values())
for part, statement in decision_frame.items():
    print(f"{part.replace('_', ' ').title()}: {statement}")

Decision: Which content item should an editor review first?
Unit: One pseudonymized content item/page at queue time.
Action: Review the page and its reason codes; choose refresh, investigate, or no change.
Wrong Call Cost: Wasted editor time for a false positive; a missed improvement opportunity for a false negative.


## 3. Quick look at the data (2-3 real numbers)

The starter file has **30,000 content-item rows**. Of those, **16,726 pages (55.8%)** have at least 500 search impressions in the 90-day window, giving a sizable group with some search volume for review prioritization. By contrast, only **17 pages** meet the narrow `stale_visible_page` rule (at least 180 days since update and at least 500 impressions). That small overlap is a useful warning: a practical review score should combine several observable reasons rather than define the entire opportunity lane with one stale-page rule. The 500-impression threshold is a screening heuristic from the starter baseline, not proof that every selected page has a reliable CTR estimate.

In [8]:
minimum_impressions = 500
has_search_volume = df["impressions_90d"] >= minimum_impressions
stale_and_visible = has_search_volume & (df["days_since_last_update"] >= 180)

print(f"Content-item rows: {len(df):,}")
print(
    f"Pages with at least {minimum_impressions:,} impressions: "
    f"{has_search_volume.sum():,} ({has_search_volume.mean():.1%})"
)
print(f"Stale-and-visible pages under the starter rule: {stale_and_visible.sum():,}")

Content-item rows: 30,000
Pages with at least 500 impressions: 16,726 (55.8%)
Stale-and-visible pages under the starter rule: 17


## 4. Careful words: what I can and can't claim

This is **decision support**, not “train a model” for its own sake: the output helps an editor spend limited review time in a more evidence-informed order, while the editor makes the final call. From this starter snapshot I can describe measured patterns and, after an honest held-out evaluation, report whether a ranking is directionally useful on similar pseudonymized content. I cannot claim that a refresh causes traffic to rise, that a page will recover, that I am predicting Google, or that the result generalizes to every client or to the full warehouse.

The current `trend_direction` and `trend_pct` summarize the same-window comparison; the starter decline label is derived from `trend_direction`. They are not independent future outcomes and must not be used as features for discovering decline risk. A future-outcome claim would require a clearly later label window, features available before that window, and validation that respects client and time boundaries.

In [9]:
claim_guardrails = {
    "output": "human review priority, not an automatic edit",
    "current_evidence": "observed signals in a trailing-90-day starter snapshot",
    "not_claimed": [
        "causal impact of a refresh",
        "guaranteed recovery or future traffic",
        "Google ranking prediction",
        "generalization beyond the evaluated population",
    ],
    "same_window_proxy_fields_excluded_from_features": {"trend_direction", "trend_pct"},
}
assert claim_guardrails["same_window_proxy_fields_excluded_from_features"].issubset(df.columns)
assert "causal impact of a refresh" in claim_guardrails["not_claimed"]
print("Claim guardrails recorded; same-window trend fields are not discovery features.")

Claim guardrails recorded; same-window trend fields are not discovery features.


## Self-check

Before submission, confirm each line honestly:

- [x] The lane is selected and justified from the guide.
- [x] The question names the decision, unit, action, and costs of both kinds of wrong call.
- [x] The quick-look code loads the starter CSV and prints three evidence metrics; the markdown interprets them.
- [x] Careful-claim boundaries and same-window label leakage are stated and checked in code.
- [x] Every code cell has been run in notebook order with no errors.
- [x] No client names, URLs, or private queries are included.
- [x] Claims are limited to observed evidence and decision support; no causal or Google-prediction claim is made.
- [ ] Commit the completed notebook to the repo and submit the repo URL on the assignment card.